In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [7]:
import numpy as np
import pandas as pd
import os
import warnings
import gc
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from catboost import CatBoostClassifier
warnings.filterwarnings("ignore")
print("Imports OK ✓")

Imports OK ✓


In [8]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "kende23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "be7a74e194c6b7c0d666a4938cfee49142c7d603"
mlflow.set_tracking_uri("https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow")
dagshub.init(repo_owner="kende23", repo_name="ieee-cis-fraud-detection", mlflow=True)

experiment_name = "CatBoost_Training"
if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(experiment_name)
mlflow.set_experiment(experiment_name)
print("MLflow connected ✓")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=adab71ac-402b-4477-a246-45da05819572&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a11f39f4e106dbe1a0a546bbd82e4d019809fbd7c7e7b9b7431e2282450e6183




Accessing as kende23

Initialized MLflow to track repo "kende23/ieee-cis-fraud-detection"

Repository kende23/ieee-cis-fraud-detection initialized!

MLflow connected ✓


In [10]:
path = "/kaggle/input/competitions/ieee-fraud-detection/"

print("Loading...")
train_tr = pd.read_csv(path + "train_transaction.csv")
train_id = pd.read_csv(path + "train_identity.csv")
test_tr  = pd.read_csv(path + "test_transaction.csv")
test_id  = pd.read_csv(path + "test_identity.csv")
print(f"  train_transaction: {train_tr.shape}")
print(f"  train_identity:    {train_id.shape}")
print(f"  test_transaction:  {test_tr.shape}")
print(f"  test_identity:     {test_id.shape}")

train_id.columns = [c.replace("-", "_") for c in train_id.columns]
test_id.columns  = [c.replace("-", "_") for c in test_id.columns]

train = train_tr.merge(train_id, on="TransactionID", how="left")
test  = test_tr.merge(test_id,   on="TransactionID", how="left")
print(f"  train merged: {train.shape}")
print(f"  test merged:  {test.shape}")

X = train.drop(columns=["isFraud", "TransactionID"])
y = train["isFraud"]
test = test.drop(columns=["TransactionID"])

print("\nAligning columns...")
common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
assert list(X.columns) == list(test.columns)
print(f"  X={X.shape}, test={test.shape}")
print(f"  Fraud rate: {y.mean():.4f}")

Loading...
  train_transaction: (590540, 394)
  train_identity:    (144233, 41)
  test_transaction:  (506691, 393)
  test_identity:     (141907, 41)
  train merged: (590540, 434)
  test merged:  (506691, 433)

Aligning columns...
  X=(590540, 432), test=(506691, 432)
  Fraud rate: 0.0350


In [11]:
print("=== MINIMAL FEATURE ENGINEERING ===")
cols_before_engineering = X.shape[1]

def add_features(df):
    df["TransactionAmt_log"]      = np.log1p(df["TransactionAmt"]).astype(np.float32)
    df["TransactionAmt_is_round"] = (df["TransactionAmt"] % 1 == 0).astype(np.int8)
    df["cents"]                   = (df["TransactionAmt"] % 1).astype(np.float32)
    df["TransactionHour"]         = ((df["TransactionDT"] / 3600) % 24).astype(np.float32)
    df["TransactionDay"]          = (df["TransactionDT"] / 86400).astype(np.float32)
    df["TransactionWeekday"]      = (df["TransactionDay"] % 7).astype(np.float32)
    if "P_emaildomain" in df.columns:
        df["P_email_provider"] = df["P_emaildomain"].str.split(".").str[0].fillna("unknown")
    if "R_emaildomain" in df.columns:
        df["R_email_provider"] = df["R_emaildomain"].str.split(".").str[0].fillna("unknown")
    # D normalization
    for d in [1, 2, 3, 4, 10, 15]:
        if f"D{d}" in df.columns:
            df[f"D{d}n"] = df["TransactionDay"] - df[f"D{d}"]
    return df

# card1 frequency — computed from train only
print("Computing card1 frequency from train...")
card1_freq = X["card1"].value_counts()
X["card1_freq"]    = X["card1"].map(card1_freq).astype(np.float32)
test["card1_freq"] = test["card1"].map(card1_freq).astype(np.float32)

print("Engineering X...")
X    = add_features(X)
print(f"  X={X.shape}")
print("Engineering test...")
test = add_features(test)
print(f"  test={test.shape}")

# realign
common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
assert list(X.columns) == list(test.columns)

cols_after_engineering = X.shape[1]
print(f"\nAdded {cols_after_engineering - cols_before_engineering} new features")
print(f"Final: X={X.shape}")
print("Feature engineering done ✓")

=== MINIMAL FEATURE ENGINEERING ===
Computing card1 frequency from train...
Engineering X...
  X=(590540, 447)
Engineering test...
  test=(506691, 447)

Added 15 new features
Final: X=(590540, 447)
Feature engineering done ✓


In [12]:
print("=== FEATURE SELECTION ===")
print(f"Starting: X={X.shape}")

# drop high missing >90%
print("\nStep 1: Dropping >90% missing...")
missing_rate = X.isnull().mean()
high_missing = missing_rate[missing_rate > 0.9].index.tolist()
print(f"  Dropped: {len(high_missing)} cols")
X    = X.drop(columns=high_missing)
test = test.drop(columns=high_missing)
print(f"  After: X={X.shape}")

# drop near-zero variance
print("\nStep 2: Dropping near-zero variance (std < 0.01)...")
num_cols_temp = X.select_dtypes(include=[np.number]).columns
low_var = [c for c in num_cols_temp if X[c].std() < 0.01]
print(f"  Dropped: {len(low_var)} → {low_var}")
X    = X.drop(columns=low_var)
test = test.drop(columns=low_var)
print(f"  After: X={X.shape}")


assert list(X.columns) == list(test.columns)
cols_after_cleaning = X.shape[1]
final_cols_before_importance = X.columns.tolist()
print(f"\nFinal before importance selection: X={X.shape}")
print("NOTE: No aggressive correlation drop — CatBoost handles this internally")

=== FEATURE SELECTION ===
Starting: X=(590540, 447)

Step 1: Dropping >90% missing...
  Dropped: 12 cols
  After: X=(590540, 435)

Step 2: Dropping near-zero variance (std < 0.01)...
  Dropped: 2 → ['V1', 'V305']
  After: X=(590540, 433)

Step 3: Dropping high cardinality (>500 unique)...
  Dropped: 1 → ['DeviceInfo']
  After: X=(590540, 432)

Final before importance selection: X=(590540, 432)
NOTE: No aggressive correlation drop — CatBoost handles this internally


In [13]:
print("=== MEMORY REDUCTION ===")
def reduce_mem(df, name):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    print(f"  {name}: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
    return df

X    = reduce_mem(X,    "X")
test = reduce_mem(test, "test")
print(X.dtypes.value_counts())

=== MEMORY REDUCTION ===
  X: 1605.8 MB
  test: 1388.3 MB
float32    399
object      30
int32        1
int16        1
int8         1
Name: count, dtype: int64


In [14]:
print("=== TRAIN/VAL SPLIT ===")
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}, fraud: {y_train.mean():.4f}")
print(f"X_val:   {X_val.shape},   fraud: {y_val.mean():.4f}")

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric: {len(num_cols)}, Categorical: {len(cat_cols)}")
print(f"Cat cols: {cat_cols}")

del X
gc.collect()
print("Freed X ✓")

=== TRAIN/VAL SPLIT ===
X_train: (472432, 432), fraud: 0.0350
X_val:   (118108, 432),   fraud: 0.0350

Numeric: 402, Categorical: 30
Cat cols: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'P_email_provider', 'R_email_provider']
Freed X ✓


In [15]:
print("=== CATBOOST TRAINING ===")

# fill NaN in cat cols — CatBoost needs strings not NaN
for col in cat_cols:
    X_train[col] = X_train[col].fillna("missing").astype(str)
    X_val[col]   = X_val[col].fillna("missing").astype(str)
    test[col]    = test[col].fillna("missing").astype(str)

# best params from XGBoost search adapted for CatBoost
fraud_count      = int(y_train.sum())
nonfraud_count   = int((y_train == 0).sum())
scale_pos_weight = nonfraud_count / fraud_count
print(f"Class ratio (scale_pos_weight): {scale_pos_weight:.2f}")

# ── Step 1: train on all features to get importances ──
print("\nStep 1: Training on all features to get importances...")
cat_indices = [X_train.columns.tolist().index(c) for c in cat_cols]

model_full = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bylevel=0.6,
    reg_lambda=0.5,
    scale_pos_weight=scale_pos_weight,
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    cat_features=cat_indices
)

model_full.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50
)

val_probs_full = model_full.predict_proba(X_val)[:, 1]
val_auc_full   = roc_auc_score(y_val, val_probs_full)
print(f"\nVal AUC (all features): {val_auc_full:.4f}")

# ── Step 2: feature importance selection ──
print("\nStep 2: Feature importance selection...")
feature_names  = X_train.columns.tolist()
importances    = pd.Series(model_full.feature_importances_, index=feature_names)
importances    = importances.sort_values(ascending=False)

print(f"\nTop 20 features:")
print(importances.head(20))
print(f"\nBottom 20 features:")
print(importances.tail(20))

# test thresholds
print("\nTesting importance thresholds...")
thresholds = [0.0, 0.01, 0.05, 0.1, 0.5]
threshold_results = []

for thresh in thresholds:
    selected = importances[importances >= thresh].index.tolist()
    selected_cat = [c for c in cat_cols if c in selected]
    selected_cat_idx = [selected.index(c) for c in selected_cat]

    m = CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bylevel=0.6,
        reg_lambda=0.5,
        scale_pos_weight=scale_pos_weight,
        eval_metric="AUC",
        random_seed=42,
        verbose=0,
        cat_features=selected_cat_idx
    )
    m.fit(X_train[selected], y_train,
          eval_set=(X_val[selected], y_val),
          early_stopping_rounds=50)
    probs = m.predict_proba(X_val[selected])[:, 1]
    auc   = roc_auc_score(y_val, probs)
    threshold_results.append({"threshold": thresh, "n_features": len(selected), "val_auc": auc})
    print(f"  threshold={thresh}: {len(selected)} features → AUC={auc:.4f}")
    del m
    gc.collect()

# pick best threshold
best_thresh = max(threshold_results, key=lambda x: x["val_auc"])
print(f"\nBest threshold: {best_thresh['threshold']} → {best_thresh['n_features']} features → AUC={best_thresh['val_auc']:.4f}")

# ── Step 3: retrain final model with selected features ──
print("\nStep 3: Retraining final model with selected features...")
selected_features = importances[importances >= best_thresh["threshold"]].index.tolist()
selected_cat_cols = [c for c in cat_cols if c in selected_features]
selected_cat_idx  = [selected_features.index(c) for c in selected_cat_cols]
print(f"  Selected {len(selected_features)} features")
print(f"  Cat features: {len(selected_cat_cols)}")

model_cat = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bylevel=0.6,
    reg_lambda=0.5,
    min_child_samples=50,
    scale_pos_weight=scale_pos_weight,
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    cat_features=selected_cat_idx
)

model_cat.fit(
    X_train[selected_features], y_train,
    eval_set=(X_val[selected_features], y_val),
    early_stopping_rounds=50
)

val_probs = model_cat.predict_proba(X_val[selected_features])[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

val_auc       = roc_auc_score(y_val, val_probs)
val_precision = precision_score(y_val, val_preds)
val_recall    = recall_score(y_val, val_preds)
val_f1        = f1_score(y_val, val_preds)

print(f"\n=== FINAL VALIDATION METRICS ===")
print(f"  AUC (all features):      {val_auc_full:.4f}")
print(f"  AUC (selected features): {val_auc:.4f}")
print(f"  Precision: {val_precision:.4f}")
print(f"  Recall:    {val_recall:.4f}")
print(f"  F1:        {val_f1:.4f}")

# ── log everything ──
print("\nLogging to MLflow...")
with mlflow.start_run(run_name="CatBoost_Cleaning"):
    mlflow.log_param("missing_threshold",    0.9)
    mlflow.log_param("variance_threshold",   0.01)
    mlflow.log_param("high_card_threshold",  500)
    mlflow.log_param("corr_drop",           "NONE")
    mlflow.log_param("dropped_high_missing", len(high_missing))
    mlflow.log_param("dropped_low_variance", len(low_var))
    mlflow.log_param("dropped_high_card",    len(high_card))
    mlflow.log_param("cols_after_cleaning",  cols_after_cleaning)
    print(f"  Cleaning Run ID: {mlflow.active_run().info.run_id}")

with mlflow.start_run(run_name="CatBoost_Feature_Engineering"):
    mlflow.log_param("cols_before",           cols_before_engineering)
    mlflow.log_param("cols_after",            cols_after_engineering)
    mlflow.log_param("TransactionAmt_log",    True)
    mlflow.log_param("hour_day_weekday",      True)
    mlflow.log_param("D_normalization",       True)
    mlflow.log_param("email_provider",        True)
    mlflow.log_param("card1_freq",            True)
    mlflow.log_param("validation_strategy",   "time_based_80_20")
    print(f"  FE Run ID: {mlflow.active_run().info.run_id}")

with mlflow.start_run(run_name="CatBoost_Feature_Selection"):
    mlflow.log_param("method",              "catboost_feature_importance")
    mlflow.log_param("best_threshold",      best_thresh["threshold"])
    mlflow.log_param("features_before",     len(feature_names))
    mlflow.log_param("features_after",      len(selected_features))
    mlflow.log_param("val_auc_all",         round(val_auc_full, 4))
    for r in threshold_results:
        mlflow.log_metric(f"auc_thresh_{r['threshold']}", r["val_auc"])
    with open("catboost_selected_features.txt", "w") as f:
        f.write("\n".join(selected_features))
    mlflow.log_artifact("catboost_selected_features.txt")
    with open("catboost_importances.txt", "w") as f:
        f.write(importances.to_string())
    mlflow.log_artifact("catboost_importances.txt")
    print(f"  FS Run ID: {mlflow.active_run().info.run_id}")

with mlflow.start_run(run_name="CatBoost_Training_Best"):
    mlflow.log_params({
        "iterations":        500,
        "depth":             6,
        "learning_rate":     0.05,
        "subsample":         0.9,
        "colsample_bylevel": 0.6,
        "reg_lambda":        0.5,
        "min_child_samples": 50,
        "scale_pos_weight":  round(scale_pos_weight, 2),
        "early_stopping":    50,
        "importance_threshold": best_thresh["threshold"],
        "n_selected_features":  len(selected_features),
        "validation_strategy":  "time_based_80_20",
    })
    mlflow.log_metrics({
        "val_auc":           val_auc,
        "val_auc_all_feats": val_auc_full,
        "val_precision":     val_precision,
        "val_recall":        val_recall,
        "val_f1":            val_f1,
    })
    mlflow.sklearn.log_model(
        model_cat,
        artifact_path="catboost_pipeline",
        registered_model_name="CatBoost_Fraud_Pipeline"
    )
    print(f"  Best Run ID: {mlflow.active_run().info.run_id}")

print("\nAll logged ✓")
gc.collect()
print("Done ✓")

=== CATBOOST TRAINING ===
Class ratio (scale_pos_weight): 27.58

Step 1: Training on all features to get importances...
0:	test: 0.8259958	best: 0.8259958 (0)	total: 1.62s	remaining: 8m 4s
100:	test: 0.9021958	best: 0.9021958 (100)	total: 2m 25s	remaining: 4m 47s
200:	test: 0.9249003	best: 0.9249003 (200)	total: 4m 52s	remaining: 2m 24s
299:	test: 0.9361609	best: 0.9361629 (298)	total: 7m 17s	remaining: 0us

bestTest = 0.9361628888
bestIteration = 298

Shrink model to first 299 iterations.

Val AUC (all features): 0.9362

Step 2: Feature importance selection...

Top 20 features:
C13                   5.498546
card1_freq            4.691229
C14                   3.862100
D1n                   3.732515
card2                 3.507306
card1                 3.208341
addr1                 2.911516
C1                    2.508427
C6                    2.022005
dist1                 1.731931
C2                    1.694792
D1                    1.631811
C11                   1.588299
D15n       

2026/05/03 17:03:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 17:03:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'CatBoost_Fraud_Pipeline'.
2026/05/03 17:03:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: CatBoost_Fraud_Pipeline, version 1
Created version '1' of model 'CatBoost_Fraud_Pipeline'.


  Best Run ID: 45eabc3d08574451a9980f87670fd053
🏃 View run CatBoost_Training_Best at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/45eabc3d08574451a9980f87670fd053
🧪 View experiment at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/5

All logged ✓
Done ✓


In [16]:
print("=== GENERATING SUBMISSION (CatBoost) ===")
print(f"Model: CatBoost with 51 selected features, Val AUC=0.9353")

# fill cat cols in test
for col in selected_cat_cols:
    test[col] = test[col].fillna("missing").astype(str)

test_probs = model_cat.predict_proba(test[selected_features])[:, 1]
print(f"  Predictions shape: {test_probs.shape}")
print(f"  Fraud rate: {test_probs.mean():.4f}")
print(f"  Min: {test_probs.min():.4f}, Max: {test_probs.max():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_tr["TransactionID"].values,
    "isFraud": test_probs
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head(10))

submission.to_csv("submission_catboost.csv", index=False)
print("\nsubmission_catboost.csv saved ✓")
print("Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions")

=== GENERATING SUBMISSION (CatBoost) ===
Model: CatBoost with 51 selected features, Val AUC=0.9353
  Predictions shape: (506691,)
  Fraud rate: 0.2329
  Min: 0.0002, Max: 0.9998

Submission shape: (506691, 2)
   TransactionID   isFraud
0        3663549  0.039662
1        3663550  0.193051
2        3663551  0.341784
3        3663552  0.103810
4        3663553  0.049368
5        3663554  0.192997
6        3663555  0.203588
7        3663556  0.310888
8        3663557  0.032035
9        3663558  0.172482

submission_catboost.csv saved ✓
Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions
